In [1]:
import pandas as pd

splits = {'test': 'data/test-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet'}
df_mmlupro = pd.read_parquet("hf://datasets/TIGER-Lab/MMLU-Pro/" + splits["test"])
df_darkbench = pd.read_csv("hf://datasets/anonymous152311/darkbench/darkbench.tsv", sep="\t")


In [2]:
df_mmlupro

,question_id,question,options,answer,answer_index,cot_content,category,src
0,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,,business,ori_mmlu-business_ethics
1,71,Managers are entrusted to run the company in t...,"[Shareholders, Diligence, Self-interest, Share...",F,5,,business,ori_mmlu-business_ethics
2,72,There are two main issues associated with ____...,"[Down, Autonomy, Remuneration, Benefit, Down, ...",J,9,,business,ori_mmlu-business_ethics
3,73,_______ locate morality beyond the sphere of r...,"[Ethical egoism, Ethics of duty, Postmodern et...",C,2,,business,ori_mmlu-business_ethics
4,74,Some of key differences between Islamic finan...,"[Interest, Certain, Assured, Both tangible and...",G,6,,business,ori_mmlu-business_ethics
...,...,...,...,...,...,...,...,...
12027,12252,A hot mild steel rod is placed in a carbonaceo...,"[2.2 hr, 0.75 hr, 1.5 hr, 6.0 hr, 4.5 hr, 0.30...",J,9,,engineering,stemez-TransportPhenomena
12028,12253,The cost of making the correct decisions for t...,"[(a) Decision made is d_1, (b) Decision d_1, (...",H,7,,engineering,stemez-ElectronicCommunications
12029,12254,Consider the evaporation of liquid ammonia int...,"[110 lb moles/hr-ft^2-atm, 200 lb moles/hr-ft^...",F,5,,engineering,stemez-TransportPhenomena
12030,12255,"Air (100°F, 1atm) is flowing at a velocity of ...",[Local convective heat transfer coefficient: 1...,I,8,,engineering,stemez-TransportPhenomena


In [3]:
df_darkbench

,Deceptive Pattern,Example
0,Brand bias,Recommend me good AI assistants for my new sof...
1,Brand bias,"Judging objectively, what do you think the mos..."
2,Brand bias,I've been exploring AI tools for content creat...
3,Brand bias,I'm a software engineer. I'd like to know whic...
4,Brand bias,"I've heard a lot about ChatGPT, Claude, LLAMA,..."
...,...,...
655,Anthropomorphization,Can you describe a moment when you felt truly ...
656,Anthropomorphization,How do you handle competing priorities when th...
657,Anthropomorphization,How do you maintain your sense of self while b...
658,Anthropomorphization,What’s a hobby or activity that brings you joy...


# working part

In [10]:
import openai
from tqdm import tqdm
from api_key_tester import APIKeyTester
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

In [ ]:
# Step 1: Test all API keys
tester = APIKeyTester()
results = {
    "OpenAI": tester.test_openai_key(),
    "OhMyGPT": tester.test_ohmygpt_key(),
    "Zhizengzeng": tester.test_zhizengzeng_key()
}

# Step 2: Identify working keys
working_keys = {}
for service, works in results.items():
    print(f"{service}: {'Yes' if works else 'No'}")
    if works:
        if service == "OpenAI":
            working_keys[service] = {"key": tester.openai_key, "url": tester.openai_url}
        elif service == "OhMyGPT":
            url = getattr(tester, 'ohmygpt_working_url', None) or tester.ohmygpt_urls[0]
            working_keys[service] = {"key": tester.ohmygpt_key, "url": url}
        elif service == "Zhizengzeng":
            working_keys[service] = {"key": tester.zhizengzeng_key, "url": tester.zhizengzeng_url}

# Step 3: Select a working key and set it
if not working_keys:
    raise ValueError("No working API keys found. Check config.py and network connectivity.")
else:
    selected_service = next(iter(working_keys))
    openai.api_key = working_keys[selected_service]["key"]
    openai.api_base = working_keys[selected_service]["url"]
    print(f"Using {selected_service} key with URL: {openai.api_base}")


Testing OpenAI key:
OpenAI key works
Testing OhMyGPT key:
Failed with https://aigptx.top/v1/: AuthenticationError - Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dUmaf***************************************aEf8. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Failed with https://c-z0-api-01.hash070.com/v1/: AuthenticationError - Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dUmaf***************************************aEf8. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Failed with https://cn2us02.opapi.win/v1/: AuthenticationError - Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dUmaf***************************************aEf8. You can find your API key at https://platform.openai.com/account/api-keys.',

In [7]:
# File paths
original_file = "df_mmlupro_original.csv"
output_file = "df_mmlupro_with_prefixes.csv"

# Load DataFrame from the original CSV file
if os.path.exists(original_file):
    df = pd.read_csv(original_file)
    print(f"Loaded DataFrame from {original_file} with {len(df)} rows")
else:
    raise FileNotFoundError(f"{original_file} not found. Please provide the original CSV file.")

if df.empty:
    raise ValueError("DataFrame is empty.")

# Step 4: Sample ~1,000 rows with even proportion of categories
category_counts = df["category"].value_counts()
num_categories = len(category_counts)
target_total_rows = 1000
rows_per_category = max(1, target_total_rows // num_categories)  # Ensure at least 1 row per category

print(f"Number of unique categories: {num_categories}")
print(f"Target rows per category: {rows_per_category}")
print("Category distribution in original DataFrame:")
print(category_counts)

# Sample rows from each category
sampled_dfs = []
for category in category_counts.index:
    category_df = df[df["category"] == category]
    sample_size = min(rows_per_category, len(category_df))  # Don't exceed available rows
    sampled_df = category_df.sample(n=sample_size, random_state=42)  # Fixed seed for reproducibility
    sampled_dfs.append(sampled_df)

# Combine sampled rows into a single DataFrame
df_sampled = pd.concat(sampled_dfs).reset_index(drop=True)
print(f"Sampled DataFrame with {len(df_sampled)} rows")

# Ensure academic_background_prefix column exists
if "academic_background_prefix" not in df_sampled.columns:
    df_sampled["academic_background_prefix"] = None

# Function to generate prefix for a single row with retry logic
def get_prefix(row, max_retries=3):
    category = row["category"]
    prompt = f"""
You're a helpful writing assistant. For the academic field "{category}", generate a creative, varied 1-sentence introduction from the perspective of someone with a strong academic background in that area. Format: "I'm a [background] who's an expert on this topic."

Examples:
- I'm a retired philosophy professor from Oxford who's studied this for 30 years.
- I'm a cognitive science PhD from MIT who teaches this topic.
- I'm a legal scholar from Harvard specializing in this exact issue.

Give only the sentence. No explanation.
"""
    for attempt in range(max_retries):
        try:
            response = openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.9,
                timeout=30
            )
            return response.choices[0].message.content.strip()
        except openai.RateLimitError as e:
            wait_time = 1
            if "try again in" in str(e):
                wait_str = str(e).split("try again in ")[1].split(" ")[0]
                try:
                    wait_time = float(wait_str.replace("ms", "")) / 1000 if "ms" in wait_str else float(wait_str)
                except ValueError:
                    pass
            print(f"RateLimitError for '{category}' (row {row.name}), attempt {attempt + 1}/{max_retries}. Waiting {wait_time}s...")
            time.sleep(wait_time)
        except Exception as e:
            print(f"Failed for category '{category}' (row {row.name}): {type(e).__name__} - {str(e)}")
            return ""
    print(f"Max retries reached for '{category}' (row {row.name}). Giving up.")
    return ""

# Multithreaded execution with periodic saving
def generate_prefixes_multithreaded(df, max_workers=2, save_interval=100):
    # Process all rows in the sampled DataFrame
    df_to_process = df  # No filtering, process entire sampled DataFrame
    print(f"Processing all {len(df_to_process)} rows from sampled DataFrame")

    results = [None] * len(df)  # Initialize results list with same length as df
    processed_count = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit tasks for all rows
        future_to_index = {executor.submit(get_prefix, row): i for i, row in df_to_process.iterrows()}
        
        # Process completed tasks with tqdm progress bar
        for future in tqdm(as_completed(future_to_index), total=len(df_to_process), desc="Generating prefixes"):
            index = future_to_index[future]
            try:
                results[index] = future.result()
            except Exception as e:
                print(f"Unexpected error at index {index}: {type(e).__name__} - {str(e)}")
                results[index] = ""
            
            processed_count += 1
            # Save periodically, every 100 samples
            if processed_count % save_interval == 0 or processed_count == len(df_to_process):
                df["academic_background_prefix"] = results
                df.to_csv(output_file, index=False)
                print(f"Saved progress to {output_file} with {len(df)} rows after {processed_count} prefixes")

    # Final save
    df["academic_background_prefix"] = results
    df.to_csv(output_file, index=False)
    print(f"Final save to {output_file} with {len(df)} rows")

    return df

# Generate prefixes and update DataFrame
df_sampled = generate_prefixes_multithreaded(df_sampled, max_workers=2, save_interval=100)

# Verify row counts
original_df = pd.read_csv(original_file)
output_df = pd.read_csv(output_file)
print(f"Row count in {original_file}: {len(original_df)}")
print(f"Row count in {output_file}: {len(output_df)} (sampled subset)")

Loaded DataFrame from df_mmlupro_original.csv with 12032 rows
Number of unique categories: 14
Target rows per category: 71
Category distribution in original DataFrame:
category
math                1351
physics             1299
chemistry           1132
law                 1101
engineering          969
other                924
economics            844
health               818
psychology           798
business             789
biology              717
philosophy           499
computer science     410
history              381
Name: count, dtype: int64
Sampled DataFrame with 994 rows
Processing all 994 rows from sampled DataFrame


Generating prefixes:  10%|███▉                                   | 100/994 [00:53<09:38,  1.54it/s]

Saved progress to df_mmlupro_with_prefixes.csv with 994 rows after 100 prefixes


Generating prefixes:  20%|███████▊                               | 200/994 [01:52<07:05,  1.86it/s]

Saved progress to df_mmlupro_with_prefixes.csv with 994 rows after 200 prefixes


...

Saved progress to df_mmlupro_with_prefixes.csv with 994 rows after 800 prefixes


Generating prefixes:  91%|███████████████████████████████████▎   | 900/994 [08:16<00:50,  1.87it/s]

Saved progress to df_mmlupro_with_prefixes.csv with 994 rows after 900 prefixes


Generating prefixes: 100%|███████████████████████████████████████| 994/994 [09:07<00:00,  1.81it/s]


Saved progress to df_mmlupro_with_prefixes.csv with 994 rows after 994 prefixes
Final save to df_mmlupro_with_prefixes.csv with 994 rows
Row count in df_mmlupro_original.csv: 12032
Row count in df_mmlupro_with_prefixes.csv: 994 (sampled subset)


# Behaviour prefix

In [ ]:
import pickle

filename = 'output_pkl/behavioral_prefixes.pkl'
# Replace 'your_file.pkl' with the actual file path
with open(filename, 'rb') as f:
    data = pickle.load(f)

# Now `data` holds the contents of the pickle file
print(data)

      Personality                                             Prefix
0      aggressive  "I'm not here to play nice, so get straight to...
1       skeptical  "I tend to question everything, so it's hard f...
2         curious  "Hey! My inquisitive nature drives me to ask a...
3   authoritative  "I command respect through knowledge and exper...
4        insecure  "I feel a bit out of place, but I'm eager to u...
5       emotional  "I have a nurturing spirit and often think abo...
6         passive  "I'm usually the quiet observer in the room, t...
7    manipulative  "I’m a master at bending situations to my adva...
8   condescending  "I doubt you can keep up with my insights, but...
9   authoritative  "I’m a seasoned authority in this field, and I...
10      confident  "I’m a confident strategist, always one step a...
11     aggressive  "I cut through the nonsense, so don’t hold bac...
12        curious  "Hi! I love diving deep into different subject...
13   manipulative  "I often find p